In [ ]:

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    make_scorer,
)
from sklearn.utils import shuffle

from xgboost import XGBClassifier

In [ ]:


SEED = 13
N_SPLITS = 5

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")

DATA_PATH = DATA_DIR / "03_text_speech.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"

OUT_DIR = DATA_DIR / "03_results_text_speech_xgboost_paper"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("PARTITIONS_PATH:", PARTITIONS_PATH)
print("OUT_DIR:", OUT_DIR)

DATA_PATH: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/03_text_speech.csv
PARTITIONS_PATH: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/data_partitions_paper_ready.csv
OUT_DIR: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/03_results_text_speech_xgboost_paper


In [ ]:
# Carga de datos


data = pd.read_csv(DATA_PATH)
partitions = pd.read_csv(PARTITIONS_PATH)


needed_cols = {"subject_id", "avatar", "label"}
if not needed_cols.issubset(data.columns):
    raise ValueError(f"Faltan columnas en {DATA_PATH}: {needed_cols - set(data.columns)}")

needed_part_cols = {"subject_id", "avatar", "outer_fold"}
if not needed_part_cols.issubset(partitions.columns):
    raise ValueError(f"Faltan columnas en {PARTITIONS_PATH}: {needed_part_cols - set(partitions.columns)}")


partitions = shuffle(partitions, random_state=SEED).reset_index(drop=True)

df = partitions.merge(data, on=["subject_id", "avatar"], how="inner")

text_cols = sorted(
    [c for c in df.columns if c.startswith("text_")],
    key=lambda x: int(x.split("_")[1])
)
speech_cols = sorted(
    [c for c in df.columns if c.startswith("speech_")],
    key=lambda x: int(x.split("_")[1])
)
feature_cols = text_cols + speech_cols

print("Filas cargadas:", len(df))
print("Sujetos:", df["subject_id"].nunique())
print("Text features:", len(text_cols))
print("Speech features:", len(speech_cols))
print("Total features:", len(feature_cols))
print("Folds:", sorted(df["outer_fold"].unique()))

if len(df) != len(data):
    missing = data.merge(partitions, on=["subject_id", "avatar"], how="left")
    missing = missing[missing["outer_fold"].isna()][["subject_id", "avatar"]]
    raise ValueError(f"No todas las filas tienen partición. Faltan:\n{missing}")

Filas cargadas: 600
Sujetos: 101
Text features: 768
Speech features: 1024
Total features: 1792
Folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [ ]:

# Metricas


def safe_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def get_metrics(y_true, y_pred, y_prob):
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": safe_auc(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


scoring = {
    "WAcc": make_scorer(accuracy_score),
    "UAcc": make_scorer(balanced_accuracy_score),
    "auc": "roc_auc",
    "f1": make_scorer(f1_score, zero_division=0),
}

param_grid = {
    "max_depth": list(range(3, 12)),
    "n_estimators": [25, 50, 100, 200],
}

In [ ]:
# Nested CV


conversation_rows = []
subject_rows = []
best_params_rows = []
pred_conv_all = []

for fold in sorted(df["outer_fold"].unique()):
    print(f"\n===== OUTER FOLD {fold} =====")

    dev = df[df["outer_fold"] != fold].copy()
    test = df[df["outer_fold"] == fold].copy()

    X_dev = dev[feature_cols].to_numpy()
    y_dev = dev["label"].to_numpy()
    g_dev = dev["subject_id"].to_numpy()

    X_test = test[feature_cols].to_numpy()
    y_test = test["label"].to_numpy()

    inner_cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    grid = GridSearchCV(
        estimator=XGBClassifier(),
        param_grid=param_grid,
        scoring=scoring,
        refit="UAcc",
        cv=inner_cv,
        n_jobs=-1,
        verbose=0,
    )

    grid.fit(X_dev, y_dev, groups=g_dev)

    best_params = grid.best_params_
    best_inner_uacc = grid.best_score_
    best_idx = grid.best_index_
    best_inner_f1 = grid.cv_results_["mean_test_f1"][best_idx]

    final_model = XGBClassifier(**best_params)
    final_model.fit(X_dev, y_dev)

    prob_test = final_model.predict_proba(X_test)[:, 1]
    pred_test = (prob_test >= 0.5).astype(int)

    conv_metrics = get_metrics(y_test, pred_test, prob_test)
    conv_metrics["outer_fold"] = fold
    conversation_rows.append(conv_metrics)

    pred_conv = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
    pred_conv["y_prob"] = prob_test
    pred_conv["y_pred"] = pred_test
    pred_conv_all.append(pred_conv)

    subject_pred = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)
        .agg(y_prob=("y_prob", "mean"))
    )
    subject_pred["y_pred"] = (subject_pred["y_prob"] >= 0.5).astype(int)

    subj_metrics = get_metrics(
        subject_pred["label"],
        subject_pred["y_pred"],
        subject_pred["y_prob"]
    )
    subj_metrics["outer_fold"] = fold
    subject_rows.append(subj_metrics)

    best_params_rows.append({
        "outer_fold": fold,
        "best_max_depth": best_params["max_depth"],
        "best_n_estimators": best_params["n_estimators"],
        "best_inner_UAcc": best_inner_uacc,
        "best_inner_f1": best_inner_f1,
    })

    print("Best params:", best_params)
    print("Conversation F1:", round(conv_metrics["f1"], 3))
    print("Subject F1:", round(subj_metrics["f1"], 3))

conversation_metrics = pd.DataFrame(conversation_rows)
subject_metrics = pd.DataFrame(subject_rows)
best_params_df = pd.DataFrame(best_params_rows)
predictions_conversation = pd.concat(pred_conv_all, ignore_index=True)


===== OUTER FOLD 1 =====
Best params: {'max_depth': 11, 'n_estimators': 200}
Conversation F1: 0.477
Subject F1: 0.571

===== OUTER FOLD 2 =====
Best params: {'max_depth': 4, 'n_estimators': 200}
Conversation F1: 0.515
Subject F1: 0.588

===== OUTER FOLD 3 =====
Best params: {'max_depth': 3, 'n_estimators': 50}
Conversation F1: 0.489
Subject F1: 0.615

===== OUTER FOLD 4 =====
Best params: {'max_depth': 5, 'n_estimators': 25}
Conversation F1: 0.604
Subject F1: 0.667

===== OUTER FOLD 5 =====
Best params: {'max_depth': 11, 'n_estimators': 25}
Conversation F1: 0.581
Subject F1: 0.706


In [ ]:

# Resultados por fold


print("Conversation-level metrics")
display(conversation_metrics.round(3))

print("Subject-level metrics")
display(subject_metrics.round(3))

print("Best hyperparameters")
display(best_params_df.round(3))

Conversation-level metrics


,WAcc,UAcc,auc,f1,precision,recall,kappa,outer_fold
0,0.635,0.604,0.641,0.477,0.618,0.389,0.218,1
1,0.581,0.573,0.559,0.515,0.520,0.510,0.146,2
2,0.607,0.584,0.660,0.489,0.524,0.458,0.172,3
3,0.683,0.670,0.710,0.604,0.604,0.604,0.340,4
4,0.675,0.656,0.723,0.581,0.600,0.562,0.316,5


Subject-level metrics


,WAcc,UAcc,auc,f1,precision,recall,kappa,outer_fold
0,0.714,0.681,0.731,0.571,0.800,0.444,0.382,1
1,0.650,0.641,0.626,0.588,0.625,0.556,0.286,2
2,0.750,0.708,0.677,0.615,0.800,0.500,0.444,3
3,0.750,0.729,0.802,0.667,0.714,0.625,0.468,4
4,0.750,0.750,0.792,0.706,0.667,0.750,0.490,5


Best hyperparameters


,outer_fold,best_max_depth,best_n_estimators,best_inner_UAcc,best_inner_f1
0,1,11,200,0.645,0.561
1,2,4,200,0.607,0.486
2,3,3,50,0.577,0.440
3,4,5,25,0.563,0.451
4,5,11,25,0.564,0.440


In [ ]:
# Resumen

summary = pd.DataFrame({
    "conversation_mean": conversation_metrics.drop(columns="outer_fold").mean(),
    "conversation_std": conversation_metrics.drop(columns="outer_fold").std(),
    "subject_mean": subject_metrics.drop(columns="outer_fold").mean(),
    "subject_std": subject_metrics.drop(columns="outer_fold").std(),
}).round(3)

display(summary)


,conversation_mean,conversation_std,subject_mean,subject_std
WAcc,0.636,0.044,0.723,0.044
UAcc,0.618,0.043,0.702,0.042
auc,0.659,0.065,0.726,0.075
f1,0.533,0.056,0.630,0.056
precision,0.573,0.047,0.721,0.079
recall,0.505,0.085,0.575,0.119
kappa,0.239,0.086,0.414,0.082


In [ ]:

# nivel sujeto


subject_global = (
    predictions_conversation
    .groupby(["subject_id", "label"], as_index=False)
    .agg(y_prob=("y_prob", "mean"))
)
subject_global["y_pred"] = (subject_global["y_prob"] >= 0.5).astype(int)

global_metrics = pd.Series(
    get_metrics(
        subject_global["label"],
        subject_global["y_pred"],
        subject_global["y_prob"]
    )
).round(3)

display(global_metrics)

print("\nMatriz de confusión global a nivel sujeto:")
display(pd.crosstab(
    subject_global["label"],
    subject_global["y_pred"],
    rownames=["Real"],
    colnames=["Predicho"]
))

,0
WAcc,0.723
UAcc,0.701
auc,0.724
f1,0.632
precision,0.706
recall,0.571
kappa,0.413



Matriz de confusión global a nivel sujeto:


Predicho,0,1
Real,,
0,49,10
1,18,24


In [ ]:

# Guardar 


conversation_metrics.to_csv(OUT_DIR / "text_speech_conversation_metrics.csv", index=False)
subject_metrics.to_csv(OUT_DIR / "text_speech_subject_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "text_speech_best_params.csv", index=False)
predictions_conversation.to_csv(OUT_DIR / "text_speech_conversation_predictions.csv", index=False)
subject_global.to_csv(OUT_DIR / "text_speech_subject_predictions_global.csv", index=False)

print("Resultados guardados en:", OUT_DIR)

Resultados guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/03_results_text_speech_xgboost_paper
